# JUNTAR 2 DATASETS Y PASARLOS A FRECUENCIA HORARIA

In [ ]:
archivo1_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_semicompletos/df__Algar_Palancia_completo.csv"               # Archivo horario con contaminantes (datetime + NO, NO2, NOx, O3, etc.)
archivo2_path = "/Users/benjamincarbonell/Desktop/i/8439X/8439X_2012_2025.csv"               # Archivo diario (fecha, tmed, velmedia, dir (0-100), sol, presMin, presMax, ...)
pattern_hourly_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_semicompletos/df__S_CEA_completo.csv"   # Archivo horario de la estación patrón (Temp, Veloc., Direc., R.Sol., Pres.)
output_csv_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/horario_unido/Algar_Palancia.csv"


In [9]:
# -*- coding: utf-8 -*-
"""
Versión corregida y robusta:
- evita errores por índices duplicados en el patrón horario
- limpia coma decimal / separadores de miles
- imputación KNN sobre variables disponibles
- quantile mapping mensual cuando existe solapamiento
- matching día→día y reconstrucción horaria
- si faltan variables en diario o patrón, las deja como NaN en el CSV final
"""
import os
import numpy as np
import pandas as pd
from datetime import timedelta
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from tqdm import tqdm

# ---------------------------
# CONFIG - EDITA RUTAS AQUÍ
# ---------------------------
archivo1_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_semicompletos/df__Algar_Palancia_completo.csv"
archivo2_path = "/Users/benjamincarbonell/Desktop/i/8439X/8439X_2012_2025.csv"
pattern_hourly_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_semicompletos/df__S_CEA_completo.csv"
output_csv_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/horario_unido/Algar_Palancia.csv"

n_neighbors_imputer = 5
seed = 42

# ---------------------------
# UTIL: lectura robusta CSV
# ---------------------------
def robust_read_csv(path, parse_dates=None, date_col=None, **kwargs):
    encodings = ['utf-8', 'latin1', 'cp1252']
    possible_seps = [',', ';', '\t', '|']
    sample = None
    try:
        with open(path, 'rb') as f:
            sample = f.read(8192).decode('utf-8', errors='ignore')
    except Exception:
        sample = None
    sep = None
    if sample:
        for s in possible_seps:
            if s in sample:
                sep = s
                break
    if sep is None:
        sep = ','
    last_err = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, sep=sep, encoding=enc, engine='python', **kwargs)
            if date_col and date_col in df.columns:
                try:
                    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
                except Exception:
                    pass
            return df
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Fallo lectura CSV {path}. Último error: {last_err}")

# ---------------------------
# UTIL: limpieza numérica (coma decimal / miles)
# ---------------------------
def clean_numeric_series(s):
    s = s.astype(str).str.strip().replace({'nan':'', 'None':''})
    sample = s.dropna().head(500).astype(str)
    use_dot_as_thousands = False
    if sample.size > 0:
        n_comma = sample.str.contains(',').sum()
        n_dot = sample.str.contains('\.').sum()
        if n_dot > 0 and n_comma > 0 and n_comma >= n_dot:
            use_dot_as_thousands = True
    s = s.str.replace(r'\s+', '', regex=True)
    if use_dot_as_thousands:
        s = s.str.replace('.', '', regex=False)
    s = s.str.replace(',', '.', regex=False)
    return pd.to_numeric(s, errors='coerce')

def clean_dataframe_numerics(df, candidate_names):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidate_names:
        for col_lower, col in cols_lower.items():
            if cand.lower() in col_lower:
                try:
                    df[col] = clean_numeric_series(df[col])
                    print(f"[clean] convertido {col}")
                except Exception as e:
                    print(f"[clean] no convertido {col}: {e}")
    return df

# ---------------------------
# UTIL: búsqueda de columnas por candidatas
# ---------------------------
def find_col(df, candidates):
    cols = df.columns
    for cand in candidates:
        for c in cols:
            if c.lower() == cand.lower():
                return c
    for cand in candidates:
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

# ---------------------------
# LEER ARCHIVOS
# ---------------------------
print("Leyendo Archivo1...")
df1 = robust_read_csv(archivo1_path)
datetime_col = find_col(df1, ['datetime', 'date', 'time', 'fecha', 'timestamp'])
if datetime_col is None:
    raise ValueError("No se encontró columna datetime en Archivo1.")
df1[datetime_col] = pd.to_datetime(df1[datetime_col], errors='coerce', utc=True)
df1 = df1.set_index(datetime_col).sort_index()
print(f"Archivo1: {df1.shape} filas. Rango: {df1.index.min()} — {df1.index.max()} (UTC)")

print("Leyendo Archivo2 (diario)...")
df2 = robust_read_csv(archivo2_path)
fecha_col = find_col(df2, ['fecha', 'date', 'day'])
if fecha_col is None:
    print("WARN: no se detectó columna fecha en Archivo2; se usará el índice si existe.")
else:
    df2[fecha_col] = pd.to_datetime(df2[fecha_col], errors='coerce').dt.date
    df2.index = pd.to_datetime(df2[fecha_col])
    df2.index.name = 'date'
print(f"Archivo2 leído: {df2.shape} filas. Rango: {df2.index.min()} — {df2.index.max()}")

print("Leyendo patrón horario...")
pattern = robust_read_csv(pattern_hourly_path)
pat_dt_col = find_col(pattern, ['datetime', 'date', 'time', 'timestamp'])
if pat_dt_col is None:
    raise ValueError("Archivo patrón horario sin columna datetime reconocible.")
pattern[pat_dt_col] = pd.to_datetime(pattern[pat_dt_col], errors='coerce', utc=True)
pattern = pattern.set_index(pat_dt_col).sort_index()
print(f"Patrón horario: {pattern.shape} filas. Rango: {pattern.index.min()} — {pattern.index.max()} (UTC)")

# ---------------------------
# LIMPIEZA NUMÉRICA
# ---------------------------
daily_numeric_candidates = ['tmed','temp','t_media','velmedia','vel','velocidad','dir','direccion','sol','r.sol','radiacion','presmin','presmax','pres_min','pres_max','pres']
print("Limpiando valores numéricos en Archivo2...")
df2 = clean_dataframe_numerics(df2, daily_numeric_candidates)

pattern_numeric_candidates = ['temp','temperature','veloc','velocidad','direc','direccion','r.sol','radiacion','pres','pressure']
print("Limpiando valores numéricos en patrón horario...")
pattern = clean_dataframe_numerics(pattern, pattern_numeric_candidates)

# ---------------------------
# RENOMBRAR COLUMNAS (intento robusto)
# ---------------------------
map_daily = {
    'tmed': ['tmed', 'temp', 't_mean', 'tmedia'],
    'velmedia': ['velmedia', 'vel', 'wind', 'wind_speed', 'velocidad'],
    'dir_raw': ['dir', 'direccion', 'dir_media', 'direction'],
    'sol': ['sol', 'r.sol', 'radiacion', 'radiacion_diaria', 'solar'],
    'presMin': ['presmin', 'pres_min', 'pmin', 'pres_minima'],
    'presMax': ['presmax', 'pres_max', 'pmax', 'pres_maxima']
}
def safe_find_and_rename(df, mapping):
    found = {}
    for target, candidates in mapping.items():
        col = find_col(df, candidates)
        if col:
            found[target] = col
    df = df.rename(columns={found[k]: k for k in found})
    return df, found

df2, found_daily = safe_find_and_rename(df2, map_daily)
print("Columnas mapeadas en Archivo2 (detectadas):", list(found_daily.keys()))

map_pat = {
    'Temp': ['temp', 'temperature', 'tmed', 't_mean', 't'],
    'Veloc': ['veloc', 'vel', 'wind', 'velmedia', 'velocidad'],
    'Direc': ['direc', 'direction', 'direccion'],
    'R.Sol': ['r.sol', 'solar', 'radiacion', 'sol'],
    'Pres': ['pres', 'pressure', 'presion']
}
pattern, found_pat = safe_find_and_rename(pattern, map_pat)
print("Columnas mapeadas en patrón horario (detectadas):", list(found_pat.keys()))

# ---------------------------
# DISPONIBILIDAD DE VARIABLES (flags)
# ---------------------------
daily = pd.DataFrame(index=df2.index)  # DataFrame diario que iremos construyendo
if 'tmed' in df2.columns:
    daily['Temp'] = df2['tmed'].astype(float)
if 'velmedia' in df2.columns:
    daily['Veloc'] = df2['velmedia'].astype(float)
if 'dir_raw' in df2.columns:
    daily['Direc'] = (df2['dir_raw'].astype(float) * 360.0 / 100.0) % 360.0
if 'sol' in df2.columns:
    daily['R.Sol'] = df2['sol'].astype(float)
if 'presMin' in df2.columns and 'presMax' in df2.columns:
    daily['Pres'] = (df2['presMin'].astype(float) + df2['presMax'].astype(float)) / 2.0

print("Columnas creadas en 'daily':", daily.columns.tolist())

# ---------------------------
# IMPUTACIÓN (KNN sobre impute_df construido dinámicamente)
# ---------------------------
print("Imputación: comprobando si se aplica...")
impute_df = pd.DataFrame(index=daily.index)
if 'Temp' in daily.columns:
    impute_df['Temp'] = daily['Temp']
if 'Veloc' in daily.columns:
    impute_df['Veloc'] = daily['Veloc']
use_uv = False
if ('Direc' in daily.columns) and ('Veloc' in daily.columns):
    use_uv = True
    deg = np.deg2rad(daily['Direc'].astype(float))
    impute_df['u'] = np.cos(deg) * daily['Veloc']
    impute_df['v'] = np.sin(deg) * daily['Veloc']
if 'R.Sol' in daily.columns:
    impute_df['R.Sol'] = daily['R.Sol']
if 'Pres' in daily.columns:
    impute_df['Pres'] = daily['Pres']

if impute_df.shape[1] >= 2 and impute_df.isnull().any().any():
    print("Aplicando KNNImputer sobre variables:", impute_df.columns.tolist())
    scaler = StandardScaler()
    knn_imputer = KNNImputer(n_neighbors=n_neighbors_imputer)
    scaled = scaler.fit_transform(impute_df)
    imputed_scaled = knn_imputer.fit_transform(scaled)
    imputed = pd.DataFrame(scaler.inverse_transform(imputed_scaled), columns=impute_df.columns, index=impute_df.index)
    if 'Temp' in imputed.columns:
        daily['Temp'] = imputed['Temp']
    if 'Veloc' in imputed.columns:
        daily['Veloc'] = imputed['Veloc']
    if 'R.Sol' in imputed.columns:
        daily['R.Sol'] = imputed['R.Sol']
    if 'Pres' in imputed.columns:
        daily['Pres'] = imputed['Pres']
    if use_uv:
        u_imp = imputed['u'].values
        v_imp = imputed['v'].values
        daily['Direc'] = (np.degrees(np.arctan2(v_imp, u_imp)) % 360)
    print("Imputación completada.")
else:
    print("No se aplica imputación KNN (pocas variables disponibles o no hay faltantes).")

# ---------------------------
# RESUMEN DIARIO DEL PATRÓN (teniendo en cuenta variables faltantes)
# ---------------------------
print("Preparando resumen diario del patrón (teniendo en cuenta variables faltantes)...")
pattern_daily = pd.DataFrame(index=pattern.resample('D').mean().index)
if 'Temp' in pattern.columns:
    pattern_daily['Temp_mean'] = pattern['Temp'].resample('D').mean()
if 'Veloc' in pattern.columns:
    pattern_daily['Veloc_mean'] = pattern['Veloc'].resample('D').mean()
if 'Direc' in pattern.columns:
    def daily_dir_mean(hourly_dir_deg):
        rad = np.deg2rad(hourly_dir_deg)
        u = np.cos(rad)
        v = np.sin(rad)
        um = np.nanmean(u)
        vm = np.nanmean(v)
        return (np.degrees(np.arctan2(vm, um)) % 360)
    pattern_daily['Direc_mean'] = pattern['Direc'].resample('D').apply(lambda s: daily_dir_mean(s.values))
if 'R.Sol' in pattern.columns:
    pattern_daily['R.Sol_sum'] = pattern['R.Sol'].resample('D').sum()
if 'Pres' in pattern.columns:
    pattern_daily['Pres_mean'] = pattern['Pres'].resample('D').mean()

# build mapping day->hourly groups (clave: usar pattern_daily.index como referencia y eliminar duplicados por hora)
pattern_by_day = {}
for day in pattern_daily.index:
    # seleccionar horas del patrón que caen en ese día [day, day+1d)
    day_start = day
    day_end = day + pd.Timedelta(days=1)
    mask = (pattern.index >= day_start) & (pattern.index < day_end)
    group = pattern.loc[mask].copy()
    if group.empty:
        continue
    # eliminar duplicados en el índice (agrupar por timestamp y promediar)
    if not group.index.is_unique:
        group = group.groupby(group.index).mean()
    pattern_by_day[day] = group

print(f"Patrón dividido por día: {len(pattern_by_day)} días disponibles en pattern_by_day.")

# añadir u/v en pattern_daily si tenemos Direc y Veloc
if 'Direc_mean' in pattern_daily.columns and 'Veloc_mean' in pattern_daily.columns:
    def deg_to_uv(deg, speed):
        rad = np.deg2rad(deg)
        u = np.cos(rad) * speed
        v = np.sin(rad) * speed
        return u, v
    p_u, p_v = deg_to_uv(pattern_daily['Direc_mean'].values, pattern_daily['Veloc_mean'].values)
    pattern_daily['u'] = p_u
    pattern_daily['v'] = p_v

# ---------------------------
# QUANTILE MAPPING (solo variables presentes en ambos)
# ---------------------------
print("Construyendo quantile mapping mensual para variables con solapamiento...")

def monthly_quantile_map(source_daily, target_daily):
    month_maps = {}
    for m in range(1,13):
        src_m = source_daily[source_daily.index.month==m].dropna()
        tgt_m = target_daily[target_daily.index.month==m].dropna()
        if len(src_m) >= 5 and len(tgt_m) >= 5:
            src_sorted = np.sort(src_m.values)
            tgt_sorted = np.sort(tgt_m.values)
            q_src = np.linspace(0,1,len(src_sorted))
            q_tgt = np.linspace(0,1,len(tgt_sorted))
            month_maps[m] = (src_sorted, q_src, tgt_sorted, q_tgt)
    if not month_maps:
        src_sorted = np.sort(source_daily.dropna().values) if source_daily.dropna().size>0 else np.array([0.0])
        tgt_sorted = np.sort(target_daily.dropna().values) if target_daily.dropna().size>0 else np.array([0.0])
        q_src = np.linspace(0,1,len(src_sorted))
        q_tgt = np.linspace(0,1,len(tgt_sorted))
        for m in range(1,13):
            month_maps[m] = (src_sorted, q_src, tgt_sorted, q_tgt)
    def mapper(dates, values):
        values = np.array(values, copy=True)
        out = values.copy()
        for i, (d, val) in enumerate(zip(dates, values)):
            m = int(pd.to_datetime(d).month)
            if m not in month_maps:
                m = sorted(month_maps.keys())[0]
            src_sorted, q_src, tgt_sorted, q_tgt = month_maps[m]
            if len(src_sorted) < 2:
                out[i] = val
                continue
            fq = np.interp(val, src_sorted, q_src, left=0.0, right=1.0)
            mapped = np.interp(fq, q_tgt, tgt_sorted)
            out[i] = mapped
        return out
    return mapper

# construir mappers solo para variables que existen en ambos
common_idx = pattern_daily.index.intersection(daily.index)
if len(common_idx) < 10:
    print("WARN: poco solapamiento entre patrón y diario; mapping puede ser inestable.")

mappers = {}
if 'Temp_mean' in pattern_daily.columns and 'Temp' in daily.columns:
    mappers['Temp'] = monthly_quantile_map(pattern_daily['Temp_mean'].loc[common_idx], daily['Temp'].loc[common_idx])
if 'Veloc_mean' in pattern_daily.columns and 'Veloc' in daily.columns:
    mappers['Veloc'] = monthly_quantile_map(pattern_daily['Veloc_mean'].loc[common_idx], daily['Veloc'].loc[common_idx])
if ('u' in pattern_daily.columns) and ('Direc' in daily.columns) and ('Veloc' in daily.columns):
    # construir u/v diarios alineados
    aligned = daily.reindex(pattern_daily.index)
    t_u = np.cos(np.deg2rad(aligned['Direc'])) * aligned['Veloc']
    t_v = np.sin(np.deg2rad(aligned['Direc'])) * aligned['Veloc']
    mappers['u'] = monthly_quantile_map(pattern_daily['u'].loc[common_idx], t_u.loc[common_idx])
    mappers['v'] = monthly_quantile_map(pattern_daily['v'].loc[common_idx], t_v.loc[common_idx])
if 'R.Sol_sum' in pattern_daily.columns and 'R.Sol' in daily.columns:
    mappers['R.Sol'] = monthly_quantile_map(pattern_daily['R.Sol_sum'].loc[common_idx], daily['R.Sol'].loc[common_idx])
if 'Pres_mean' in pattern_daily.columns and 'Pres' in daily.columns:
    mappers['Pres'] = monthly_quantile_map(pattern_daily['Pres_mean'].loc[common_idx], daily['Pres'].loc[common_idx])

print("Mappers construidos para:", list(mappers.keys()))

# ---------------------------
# Preparar mapped_pattern_daily parcialmente (solo para variables mapeadas)
# ---------------------------
mapped_pattern_daily = pattern_daily.copy()
if 'Temp' in mappers:
    mapped_pattern_daily['Temp_mean_mapped'] = mappers['Temp'](pattern_daily.index, pattern_daily['Temp_mean'].values)
if 'Veloc' in mappers:
    mapped_pattern_daily['Veloc_mean_mapped'] = mappers['Veloc'](pattern_daily.index, pattern_daily['Veloc_mean'].values)
if 'u' in mappers and 'v' in mappers:
    mapped_pattern_daily['u_mapped'] = mappers['u'](pattern_daily.index, pattern_daily['u'].values)
    mapped_pattern_daily['v_mapped'] = mappers['v'](pattern_daily.index, pattern_daily['v'].values)
    mapped_pattern_daily['Veloc_mean_mapped'] = np.sqrt(mapped_pattern_daily['u_mapped']**2 + mapped_pattern_daily['v_mapped']**2)
    mapped_pattern_daily['Direc_mean_mapped'] = (np.degrees(np.arctan2(mapped_pattern_daily['v_mapped'], mapped_pattern_daily['u_mapped'])) % 360)
if 'R.Sol' in mappers:
    mapped_pattern_daily['R.Sol_sum_mapped'] = mappers['R.Sol'](pattern_daily.index, pattern_daily['R.Sol_sum'].values)
if 'Pres' in mappers:
    mapped_pattern_daily['Pres_mean_mapped'] = mappers['Pres'](pattern_daily.index, pattern_daily['Pres_mean'].values)

mapped_for_matching_cols = [c for c in ['Temp_mean_mapped','Veloc_mean_mapped','Direc_mean_mapped','R.Sol_sum_mapped','Pres_mean_mapped'] if c in mapped_pattern_daily.columns]
mapped_for_matching = mapped_pattern_daily[mapped_for_matching_cols].copy()
# renombrar columnas a nombres cortos sin "_mapped"
mapped_for_matching.columns = [c.replace('_mapped','') for c in mapped_for_matching.columns]

# ---------------------------
# Matching + reconstrucción (reconstruimos solo variables disponibles)
# ---------------------------
print("Reconstruyendo horas por día (solo variables disponibles se reconstruyen).")

def deg_to_uv(deg, speed):
    rad = np.deg2rad(deg)
    u = np.cos(rad) * speed
    v = np.sin(rad) * speed
    return u, v

def find_best_pattern_day(target_date, target_row, mapped_df):
    # If no matching variables available, fallback to nearest day
    if mapped_df.shape[1] == 0:
        return min(list(pattern_by_day.keys()), key=lambda d: abs((d - pd.to_datetime(target_date)).days))
    month = pd.to_datetime(target_date).month
    candidates = mapped_df[mapped_df.index.month == month]
    if len(candidates) == 0:
        candidates = mapped_df
    # build target vector only for columns present
    tgt_vec = []
    for col in candidates.columns:
        # target_row may be a Series with daily columns (Temp, Veloc, Direc, R.Sol, Pres)
        val = target_row.get(col.replace('_mean',''), np.nan) if hasattr(target_row, 'get') else np.nan
        tgt_vec.append(val if not pd.isna(val) else np.nan)
    tgt_vec = np.array(tgt_vec, dtype=float)
    cand_matrix = []
    for idx, row in candidates.iterrows():
        cand_vec = np.array(row.values, dtype=float)
        cand_matrix.append({'idx': idx, 'vec': cand_vec})
    all_vecs = np.vstack([c['vec'] for c in cand_matrix])
    stds = np.nanstd(all_vecs, axis=0)
    stds[stds==0] = 1.0
    best_score = np.inf
    best_idx = None
    for c in cand_matrix:
        norm_diff = (c['vec'] - tgt_vec) / stds
        score = np.nansum(np.nan_to_num(norm_diff**2))
        if score < best_score:
            best_score = score
            best_idx = c['idx']
    return best_idx

def reconstruct_hourly_for_day(target_date, target_row, matched_pattern_day_hourly):
    # target_date: Timestamp (day)
    day_start = pd.to_datetime(target_date).tz_localize('UTC') if pd.to_datetime(target_date).tzinfo is None else pd.to_datetime(target_date)
    desired_idx = pd.date_range(start=day_start, periods=24, freq='H', tz='UTC')
    hourly_df = pd.DataFrame(index=desired_idx)
    # Defensive: if matched_pattern_day_hourly is empty, return all-NaN DF
    if matched_pattern_day_hourly is None or matched_pattern_day_hourly.empty:
        for col in ['Temp','Veloc','Direc','R.Sol','Pres']:
            hourly_df[col] = np.nan
        return hourly_df
    ph = matched_pattern_day_hourly.copy()
    # eliminar duplicados en índice (agregar por timestamp si existen)
    if not ph.index.is_unique:
        ph = ph.groupby(ph.index).mean()
    # Asegurar que el índice sea tz-aware en UTC (coherente con desired_idx)
    if getattr(ph.index, 'tz', None) is None:
        # si ph es naive asumimos UTC (porque pattern se leyó en UTC); convertir a tz-aware UTC
        ph.index = ph.index.tz_localize('UTC')
    else:
        # si tiene tz distinta, convertir a UTC
        try:
            ph.index = ph.index.tz_convert('UTC')
        except Exception:
            # en caso raro, intentamos remover tz y localizar a UTC
            ph.index = ph.index.tz_localize(None).tz_localize('UTC')
    # ahora reindex usando nearest con tolerancia (ya no debe dar error por duplicados)
    try:
        ph = ph.reindex(desired_idx, method='nearest', tolerance=pd.Timedelta('60m'))
    except Exception:
        # fallback: simple reindex + interpolate
        ph = ph.reindex(desired_idx)
    ph = ph.reindex(desired_idx).interpolate().ffill().bfill()
    # Ahora reconstruir variable por variable solo si están presentes
    # Temp: additive
    if 'Temp' in ph.columns and 'Temp' in daily.columns:
        pat_daily_mean_temp = ph['Temp'].mean()
        tgt_temp = target_row.get('Temp', np.nan)
        hourly_df['Temp'] = ph['Temp'] + (tgt_temp - pat_daily_mean_temp) if not pd.isna(tgt_temp) else np.nan
    else:
        hourly_df['Temp'] = np.nan
    # Veloc: multiplicative
    if 'Veloc' in ph.columns and 'Veloc' in daily.columns:
        pat_daily_mean_vel = ph['Veloc'].mean()
        tgt_vel = target_row.get('Veloc', np.nan)
        scale_vel = 1.0
        if (not pd.isna(pat_daily_mean_vel)) and pat_daily_mean_vel != 0 and (not pd.isna(tgt_vel)):
            scale_vel = tgt_vel / pat_daily_mean_vel
        hourly_df['Veloc'] = ph['Veloc'] * scale_vel
    else:
        hourly_df['Veloc'] = np.nan
    # Direction: reconstruct u/v only if both direction and veloc present in daily and in pattern
    if ('Direc' in daily.columns) and ('Veloc' in daily.columns) and ('Direc' in ph.columns) and ('Veloc' in ph.columns):
        pat_u = np.cos(np.deg2rad(ph['Direc'])) * ph['Veloc']
        pat_v = np.sin(np.deg2rad(ph['Direc'])) * ph['Veloc']
        tgt_dir = target_row.get('Direc', np.nan)
        tgt_vel = target_row.get('Veloc', np.nan)
        if pd.isna(tgt_dir) or pd.isna(tgt_vel):
            hourly_df['Direc'] = np.nan
        else:
            tgt_u, tgt_v = deg_to_uv(tgt_dir, tgt_vel)
            # use computed hourly Veloc if available to scale u/v
            if hourly_df['Veloc'].notna().any():
                # avoid divide by zero
                ph_mean_vel = np.nanmean(ph['Veloc']) if np.nanmean(ph['Veloc']) != 0 else 1.0
                scale = np.nanmean(hourly_df['Veloc']) / ph_mean_vel
                u_scaled = pat_u * scale
                v_scaled = pat_v * scale
            else:
                u_scaled = pat_u
                v_scaled = pat_v
            delta_u = tgt_u - np.nanmean(u_scaled)
            delta_v = tgt_v - np.nanmean(v_scaled)
            u_final = u_scaled + delta_u
            v_final = v_scaled + delta_v
            hourly_df['Direc'] = (np.degrees(np.arctan2(v_final, u_final)) % 360)
    else:
        hourly_df['Direc'] = np.nan
    # R.Sol: multiplicative to match daily sum
    if 'R.Sol' in ph.columns and 'R.Sol' in daily.columns:
        pat_sum = ph['R.Sol'].sum()
        tgt_sum = target_row.get('R.Sol', np.nan)
        if (not pd.isna(pat_sum)) and pat_sum != 0 and (not pd.isna(tgt_sum)):
            hourly_df['R.Sol'] = ph['R.Sol'] * (tgt_sum / pat_sum)
        else:
            hourly_df['R.Sol'] = np.nan
    else:
        hourly_df['R.Sol'] = np.nan
    # Pres: additive
    if 'Pres' in ph.columns and 'Pres' in daily.columns:
        tgt_pres = target_row.get('Pres', np.nan)
        hourly_df['Pres'] = ph['Pres'] + (tgt_pres - ph['Pres'].mean()) if not pd.isna(tgt_pres) else np.nan
    else:
        hourly_df['Pres'] = np.nan
    return hourly_df

# ---------------------------
# Construir índice horario objetivo y lista de días diarios a procesar
# ---------------------------
start_date = df1.index.min().date()
end_date = df1.index.max().date()
all_hours_index = pd.date_range(
    start=pd.to_datetime(start_date).tz_localize('UTC'),
    end=pd.to_datetime(end_date).tz_localize('UTC') + pd.Timedelta(hours=23),
    freq='H',
    tz='UTC'
)
# daily dates available (index entries) filtered by date range
daily_dates_series = pd.Series(pd.to_datetime(daily.index).date, index=daily.index)
daily_dates_to_process = daily_dates_series[(daily_dates_series >= start_date) & (daily_dates_series <= end_date)].index
print(f"Días diarios a procesar: {len(daily_dates_to_process)} (solo días presentes en Archivo2 dentro del rango de Archivo1)")

# ---------------------------
# LOOP: match day -> pattern day -> reconstruct hourly
# ---------------------------
hourly_list = []
pattern_days_list = list(pattern_by_day.keys())

# mapped_for_matching preparado previamente (puede estar vacío)
mapped_for_matching_df = mapped_for_matching if 'mapped_for_matching' in locals() and not mapped_for_matching.empty else pd.DataFrame()

for day in tqdm(daily_dates_to_process, desc="Procesando días"):
    target_row = daily.loc[day]
    try:
        best_pat_date = find_best_pattern_day(day, target_row, mapped_for_matching_df)
    except Exception:
        best_pat_date = min(pattern_days_list, key=lambda d: abs((d - pd.to_datetime(day)).days))
    pat_hourly = pattern_by_day.get(best_pat_date, None)
    if pat_hourly is None:
        nearest = min(pattern_by_day.keys(), key=lambda d: abs((d - best_pat_date).days))
        pat_hourly = pattern_by_day[nearest]
    rec_hourly = reconstruct_hourly_for_day(day, target_row, pat_hourly)
    hourly_list.append(rec_hourly)

# concatenar resultados
if len(hourly_list) > 0:
    hourly_meteorology = pd.concat(hourly_list)
    # en caso de índices duplicados tras concatenar (teórico si se solapan días), agrupar por índice y promediar
    if not hourly_meteorology.index.is_unique:
        hourly_meteorology = hourly_meteorology.groupby(hourly_meteorology.index).mean()
    # reindexar al índice horario objetivo (algunas horas pueden faltar -> NaN)
    hourly_meteorology = hourly_meteorology.reindex(all_hours_index)
else:
    hourly_meteorology = pd.DataFrame(index=all_hours_index, columns=['Temp','Veloc','Direc','R.Sol','Pres'])

print(f"Reconstrucción horaria finalizada: {hourly_meteorology.shape} filas")

# ---------------------------
# UNIR con Archivo1 (mantener contaminantes originales)
# ---------------------------
print("Uniendo meteorología reconstruida con Archivo1...")
result = df1.copy()
for col in ['Veloc','Direc','Temp','R.Sol','Pres']:
    if col in hourly_meteorology.columns:
        # align by index; hourly_meteorology index está en UTC tz-aware, result index también lo está
        result[col] = hourly_meteorology[col]
    else:
        result[col] = np.nan

# rellenar huecos cortos (limit=3) pero no crear datos donde no había info
result[['Veloc','Direc','Temp','R.Sol','Pres']] = result[['Veloc','Direc','Temp','R.Sol','Pres']].interpolate(limit=3).ffill().bfill()

# seleccionar columnas de salida (mantener contaminantes si existen)
requested_cols = ['NO','NO2','NOx','O3','Veloc','Direc','Temp','R.Sol','Pres']
existing = [c for c in requested_cols if c in result.columns]
result_out = result[existing].copy()

# Guardar CSV
os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
result_out.to_csv(output_csv_path, index=True, date_format='%Y-%m-%d %H:%M:%S%z')
print(f"CSV horario guardado en: {output_csv_path}")

# Informe de disponibilidad
print("Resumen final de disponibilidad (columnas en CSV final):")
for c in requested_cols:
    present = c in result_out.columns and result_out[c].notna().any()
    print(f" - {c}: {'Tiene datos (algunos no-NaN)' if present else 'Solo NaN o no presente'}")

print("Proceso completado.")


Leyendo Archivo1...
Archivo1: (116634, 5) filas. Rango: 2012-03-12 11:00:00+00:00 — 2025-12-31 23:00:00+00:00 (UTC)
Leyendo Archivo2 (diario)...
Archivo2 leído: (5077, 20) filas. Rango: 2012-01-01 00:00:00 — 2025-12-31 00:00:00
Leyendo patrón horario...
Patrón horario: (130030, 10) filas. Rango: 2012-01-01 00:00:00+00:00 — 2025-12-31 23:00:00+00:00 (UTC)
Limpiando valores numéricos en Archivo2...
[clean] convertido tmed
[clean] convertido velmedia
[clean] convertido velmedia
[clean] convertido dir
Limpiando valores numéricos en patrón horario...
[clean] convertido Temp.
[clean] convertido Veloc.
[clean] convertido Direc.
[clean] convertido R.Sol.
[clean] convertido Pres.
Columnas mapeadas en Archivo2 (detectadas): ['tmed', 'velmedia', 'dir_raw']
Columnas mapeadas en patrón horario (detectadas): ['Temp', 'Veloc', 'Direc', 'R.Sol', 'Pres']
Columnas creadas en 'daily': ['Temp', 'Veloc', 'Direc']
Imputación: comprobando si se aplica...
Aplicando KNNImputer sobre variables: ['Temp', 'Veloc'

/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_39837/492350270.py:261: RuntimeWarning: Mean of empty slice
  um = np.nanmean(u)
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_39837/492350270.py:262: RuntimeWarning: Mean of empty slice
  vm = np.nanmean(v)
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_39837/492350270.py:528: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  all_hours_index = pd.date_range(


Patrón dividido por día: 5094 días disponibles en pattern_by_day.
Construyendo quantile mapping mensual para variables con solapamiento...
WARN: poco solapamiento entre patrón y diario; mapping puede ser inestable.
Mappers construidos para: ['Temp', 'Veloc', 'u', 'v']
Reconstruyendo horas por día (solo variables disponibles se reconstruyen).
Días diarios a procesar: 5006 (solo días presentes en Archivo2 dentro del rango de Archivo1)


Procesando días:   0%|          | 0/5006 [00:00<?, ?it/s]/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_39837/492350270.py:431: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  desired_idx = pd.date_range(start=day_start, periods=24, freq='H', tz='UTC')
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_39837/492350270.py:498: RuntimeWarning: Mean of empty slice
  delta_u = tgt_u - np.nanmean(u_scaled)
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_39837/492350270.py:499: RuntimeWarning: Mean of empty slice
  delta_v = tgt_v - np.nanmean(v_scaled)
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_39837/492350270.py:431: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  desired_idx = pd.date_range(start=day_start, periods=24, freq='H', tz='UTC')
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_39837/492350270.py:498: RuntimeWarning: Mean

Reconstrucción horaria finalizada: (121032, 5) filas
Uniendo meteorología reconstruida con Archivo1...
CSV horario guardado en: /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/horario_unido/Algar_Palancia.csv
Resumen final de disponibilidad (columnas en CSV final):
 - NO: Tiene datos (algunos no-NaN)
 - NO2: Tiene datos (algunos no-NaN)
 - NOx: Tiene datos (algunos no-NaN)
 - O3: Tiene datos (algunos no-NaN)
 - Veloc: Tiene datos (algunos no-NaN)
 - Direc: Tiene datos (algunos no-NaN)
 - Temp: Tiene datos (algunos no-NaN)
 - R.Sol: Solo NaN o no presente
 - Pres: Solo NaN o no presente
Proceso completado.
